# NIFTY Morning Signal

**Two-phase workflow:**

| Phase | When | Cells to run | What you get |
|---|---|---|---|
| Pre-market | Before 9:15 AM | 1 → 2 → 3 → 4 | Signal from overnight data (no gap yet) |
| Post-open  | At/after 9:15 AM | 5 → 6 | Final signal including actual gap |

No manual input needed for Phase 1. Only Cell 5 requires one number.

In [1]:
# ── Cell 1: Imports & Config (run once) ───────────────────────────────────────
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import date, timedelta
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE        = Path.cwd()
SIGNALS_CSV = BASE / 'v2' / 'v2_reliable_signals.csv'

# ── Thresholds — must match py_backtest.py exactly ────────────────────────────
GAP_THRESHOLD       = 0.0015   # 0.15%  gap up/down
GAP_LARGE_THRESHOLD = 0.0050   # 0.50%  strong gap  ← matches training threshold
VIX_RISING_THRESHOLD = 0.03    # 3%     VIX rising  ← matches training threshold
VIX_SPIKE_THRESHOLD = 0.05     # 5%     VIX spike
BASE_RATE           = 54.5     # historical base rate: 54.5% of sessions are DOWN

# ── Trade parameters — matches best backtest result ───────────────────────────
STOP_LOSS_PCT      = 0.40      # exit if option premium drops 40%
PROFIT_TARGET_PCT  = 0.40      # exit if option premium rises 40%

W = 70   # output width

print('Config loaded.')
print(f'  SL={STOP_LOSS_PCT:.0%} of premium  |  TP={PROFIT_TARGET_PCT:.0%} of premium')
print(f'  Gap Large = {GAP_LARGE_THRESHOLD:.2%}  |  VIX Rising = >{VIX_RISING_THRESHOLD:.0%}')

Config loaded.
  SL=40% of premium  |  TP=40% of premium
  Gap Large = 0.50%  |  VIX Rising = >3%


In [2]:
# ── Cell 2: Fetch global market data — FULLY AUTOMATIC ────────────────────────
today = date.today()
start = str(today - timedelta(days=12))
end   = str(today + timedelta(days=1))

print(f'Fetching market data for {today} ...', end=' ', flush=True)

TICKERS = {
    'SP500'  : '^GSPC',
    'SGX'    : 'NKD=F',    # Nikkei futures — directional proxy for NIFTY
    'DAX'    : '^GDAXI',
    'VIX_US' : '^VIX',
    'NIFTY'  : '^NSEI',
}

raw = {}
for name, ticker in TICKERS.items():
    df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df[df.index.date < today].copy()
    raw[name] = df

print('done.\n')

def _ret(name):
    df = raw[name]
    if len(df) < 2:
        return None
    return float((df['Close'].iloc[-1] - df['Close'].iloc[-2]) / df['Close'].iloc[-2])

def _close(name):
    df = raw[name]
    return float(df['Close'].iloc[-1]) if len(df) > 0 else None

sp500_ret   = _ret('SP500')
sgx_ret     = _ret('SGX')
dax_ret     = _ret('DAX')
vix_ret     = _ret('VIX_US')
nifty_ret   = _ret('NIFTY')
nifty_close = _close('NIFTY')
vix_level   = _close('VIX_US')

print(f'  NIFTY prev close   : {nifty_close:>10,.1f}')
print(f'  S&P 500 yesterday  : {sp500_ret:>+10.2%}')
print(f'  SGX/Nikkei futures : {sgx_ret:>+10.2%}')
print(f'  DAX yesterday      : {dax_ret:>+10.2%}')
print(f'  VIX yesterday      : {vix_ret:>+10.2%}  (level: {vix_level:.1f})')
print(f'  Prev India return  : {nifty_ret:>+10.2%}')

Fetching market data for 2026-04-01 ... done.

  NIFTY prev close   :   22,331.4
  S&P 500 yesterday  :     +2.91%
  SGX/Nikkei futures :     +3.85%
  DAX yesterday      :     +0.52%
  VIX yesterday      :    -17.51%  (level: 25.2)
  Prev India return  :     -2.14%


In [3]:
# ── Cell 3: Compute overnight signals (no gap yet) ────────────────────────────
if not nifty_close or nifty_close == 0:
    raise ValueError('NIFTY previous close unavailable — market holiday or yfinance issue.')

# Gap signals are unknown pre-market — set to False, updated in Cell 6
signals = {
    'Gap Up'          : False,
    'Gap Up Strong'   : False,
    'Gap Down'        : False,
    'Prev India UP'   : nifty_ret is not None and nifty_ret  > 0,
    'Prev India DOWN' : nifty_ret is not None and nifty_ret  < 0,
    'US UP'           : sp500_ret is not None and sp500_ret  > 0,
    'US DOWN'         : sp500_ret is not None and sp500_ret  < 0,
    'SGX UP'          : sgx_ret   is not None and sgx_ret    > 0,
    'SGX DOWN'        : sgx_ret   is not None and sgx_ret    < 0,
    'DAX UP'          : dax_ret   is not None and dax_ret    > 0,
    'VIX Rising'      : vix_ret   is not None and vix_ret    > VIX_RISING_THRESHOLD,  # >3%
    'VIX Falling'     : vix_ret   is not None and vix_ret    < 0,
    'VIX Spike'       : vix_ret   is not None and vix_ret    > VIX_SPIKE_THRESHOLD,
}

active_overnight = [k for k, v in signals.items() if v]

print('Overnight signals (gap unknown):')
for s in active_overnight:
    print(f'  ✓  {s}')
if not active_overnight:
    print('  (none active)')

Overnight signals (gap unknown):
  ✓  Prev India DOWN
  ✓  US UP
  ✓  SGX UP
  ✓  DAX UP
  ✓  VIX Falling


In [4]:
# ── Cell 4: PRE-MARKET SIGNAL (run before 9:15 AM) ────────────────────────────
# Uses the same top-3 bearish + top-3 bullish as the backtest.
# Gap-based combos are excluded here (gap is unknown pre-market).

if not SIGNALS_CSV.exists():
    raise FileNotFoundError(f'{SIGNALS_CSV} not found. Run v2/v2_india_global.ipynb first.')

reliable = pd.read_csv(SIGNALS_CSV)

# ── Replicate py_backtest.py signal selection exactly ─────────────────────────
top_bearish = (reliable[reliable['P_Down'] > BASE_RATE]
               .sort_values('Edge', ascending=False)
               .head(3)
               .reset_index(drop=True))

top_bullish = (reliable[reliable['P_Down'] < (100 - BASE_RATE)]
               .sort_values('P_Down', ascending=True)
               .head(3)
               .reset_index(drop=True))

bear_combos = list(top_bearish['Signal'])
bull_combos = list(top_bullish['Signal'])

GAP_SIGNALS = {'Gap Up', 'Gap Up Strong', 'Gap Down'}

def _has_gap(combo_str):
    return any(g in [s.strip() for s in combo_str.split('+')] for g in GAP_SIGNALS)

def _combo_fires(combo_str):
    return all(signals.get(s.strip(), False) for s in combo_str.split('+'))

# Pre-market: only non-gap combos can fire
bear_fired = [c for c in bear_combos if not _has_gap(c) and _combo_fires(c)]
bull_fired  = [c for c in bull_combos if not _has_gap(c) and _combo_fires(c)]

# ── Print helper ───────────────────────────────────────────────────────────────
def _print_signal(bear_fired, bull_fired, label, show_top3=True):
    if show_top3:
        print('  Top-3 bearish combos being monitored:')
        for c in bear_combos:
            tag = '✓ ACTIVE' if c in bear_fired else '✗'
            print(f'    {tag}  {c}')
        print('  Top-3 bullish combos being monitored:')
        for c in bull_combos:
            tag = '✓ ACTIVE' if c in bull_fired else '✗'
            print(f'    {tag}  {c}')
        print()

    print('=' * W)
    if not bear_fired and not bull_fired:
        print(f'  {label}:  NEUTRAL  —  SIT OUT')
        print(f'  None of the top-3 bear/bull combos are active.')
    elif bear_fired and not bull_fired:
        row = top_bearish[top_bearish['Signal'] == bear_fired[0]].iloc[0]
        print(f'  {label}:  ▼  BEARISH  (Buy 1-OTM PUT)')
        print(f'  Combo  : {bear_fired[0]}')
        print(f'  P(DOWN): {row["P_Down"]:.1f}%  |  Edge: +{row["Edge"]:.1f}%  |  {row["Sig"]}  N={int(row["N"])}')
    elif bull_fired and not bear_fired:
        row = top_bullish[top_bullish['Signal'] == bull_fired[0]].iloc[0]
        print(f'  {label}:  ▲  BULLISH  (Buy 1-OTM CALL)')
        print(f'  Combo  : {bull_fired[0]}')
        print(f'  P(UP)  : {100-row["P_Down"]:.1f}%  |  Edge: +{abs(row["Edge"]):.1f}%  |  {row["Sig"]}  N={int(row["N"])}')
    else:
        print(f'  {label}:  ⚡ CONFLICT  —  SIT OUT  (both bear & bull fire)')
    print('=' * W)

print('=' * W)
print(f"  PRE-MARKET  ·  {today.strftime('%d %b %Y, %A')}  ·  (gap unknown)")
print('=' * W)
print()
_print_signal(bear_fired, bull_fired, label='PRE-MARKET SIGNAL')
print()
print('  → Wait for 9:15 AM, enter actual NIFTY open in Cell 5, then run Cell 6.')

  PRE-MARKET  ·  01 Apr 2026, Wednesday  ·  (gap unknown)

  Top-3 bearish combos being monitored:
    ✗  Gap Up + Prev India DOWN + US UP + SGX UP
    ✗  Gap Up + Prev India DOWN + SGX UP + DAX UP
    ✗  Gap Up + Prev India DOWN + SGX UP
  Top-3 bullish combos being monitored:
    ✗  Gap Down + US UP
    ✗  Prev India DOWN + US UP + SGX DOWN
    ✗  Gap Down + SGX UP

  PRE-MARKET SIGNAL:  NEUTRAL  —  SIT OUT
  None of the top-3 bear/bull combos are active.

  → Wait for 9:15 AM, enter actual NIFTY open in Cell 5, then run Cell 6.


In [5]:
# ── Cell 5: MANUAL INPUT — enter actual NIFTY open at/after 9:15 AM ──────────
#
#  Look at the NIFTY 50 index on your broker screen the moment market opens.
#  Use the price at 9:15 AM (first candle open).
#
NIFTY_ACTUAL_OPEN = 22450    # ← ENTER ACTUAL NIFTY OPEN PRICE HERE
#
# ─────────────────────────────────────────────────────────────────────────────
print(f'Actual NIFTY open set to: {NIFTY_ACTUAL_OPEN:,.0f}')
print('Run Cell 6 for the final signal.')

Actual NIFTY open set to: 22,450
Run Cell 6 for the final signal.


In [6]:
# ── Cell 6: FINAL SIGNAL — includes actual gap (run at/after 9:15 AM) ─────────
gap = (NIFTY_ACTUAL_OPEN - nifty_close) / nifty_close

if gap > GAP_LARGE_THRESHOLD:
    gap_label = f'GAP UP STRONG ({gap:+.2%})'
elif gap > GAP_THRESHOLD:
    gap_label = f'GAP UP ({gap:+.2%})'
elif gap < -GAP_THRESHOLD:
    gap_label = f'GAP DOWN ({gap:+.2%})'
else:
    gap_label = f'FLAT ({gap:+.2%}) — no gap signal'

# Update signals dict with actual gap
signals['Gap Up']        = gap >  GAP_THRESHOLD
signals['Gap Up Strong'] = gap >  GAP_LARGE_THRESHOLD
signals['Gap Down']      = gap < -GAP_THRESHOLD

all_active = [k for k, v in signals.items() if v]
print(f'Gap : {NIFTY_ACTUAL_OPEN:,.0f} vs prev close {nifty_close:,.0f}  →  {gap_label}')
print(f'Active signals: {", ".join(all_active)}')
print()

# Check top-3 bear + top-3 bull (all combos now eligible including gap)
bear_fired_final = [c for c in bear_combos if _combo_fires(c)]
bull_fired_final  = [c for c in bull_combos if _combo_fires(c)]

print('=' * W)
print(f"  FINAL SIGNAL  ·  {today.strftime('%d %b %Y, %A')}  ·  Open: {NIFTY_ACTUAL_OPEN:,.0f}  ({gap:+.2%})")
print('=' * W)
print()
_print_signal(bear_fired_final, bull_fired_final, label='FINAL SIGNAL')

# ── Trade execution guide ──────────────────────────────────────────────────────
print()
print('  EXECUTION GUIDE:')
print(f'    · Strike       : 1-OTM weekly expiry  (nearest Thursday)')

if bear_fired_final and not bull_fired_final:
    atm_approx = round(NIFTY_ACTUAL_OPEN / 50) * 50
    strike     = atm_approx - 50
    print(f'    · ATM ≈ {atm_approx:,}  →  Buy PE {strike}  (1-OTM put)')
elif bull_fired_final and not bear_fired_final:
    atm_approx = round(NIFTY_ACTUAL_OPEN / 50) * 50
    strike     = atm_approx + 50
    print(f'    · ATM ≈ {atm_approx:,}  →  Buy CE {strike}  (1-OTM call)')

print(f'    · Stop Loss    : Exit if option premium falls {STOP_LOSS_PCT:.0%} from entry')
print(f'    · Profit Target: Exit if option premium rises {PROFIT_TARGET_PCT:.0%} from entry')
print(f'    · Hard exit    : Close trade by 10:15 AM regardless of P&L')
print(f'    · Lot size     : 1 lot = 75 units  (NIFTY weekly)')

Gap : 22,450 vs prev close 22,331  →  GAP UP STRONG (+0.53%)
Active signals: Gap Up, Gap Up Strong, Prev India DOWN, US UP, SGX UP, DAX UP, VIX Falling

  FINAL SIGNAL  ·  01 Apr 2026, Wednesday  ·  Open: 22,450  (+0.53%)

  Top-3 bearish combos being monitored:
    ✓ ACTIVE  Gap Up + Prev India DOWN + US UP + SGX UP
    ✓ ACTIVE  Gap Up + Prev India DOWN + SGX UP + DAX UP
    ✓ ACTIVE  Gap Up + Prev India DOWN + SGX UP
  Top-3 bullish combos being monitored:
    ✗  Gap Down + US UP
    ✗  Prev India DOWN + US UP + SGX DOWN
    ✗  Gap Down + SGX UP

  FINAL SIGNAL:  ▼  BEARISH  (Buy 1-OTM PUT)
  Combo  : Gap Up + Prev India DOWN + US UP + SGX UP
  P(DOWN): 73.8%  |  Edge: +19.3%  |  **  N=65

  EXECUTION GUIDE:
    · Strike       : 1-OTM weekly expiry  (nearest Thursday)
    · ATM ≈ 22,450  →  Buy PE 22400  (1-OTM put)
    · Stop Loss    : Exit if option premium falls 40% from entry
    · Profit Target: Exit if option premium rises 40% from entry
    · Hard exit    : Close trade by 10: